# Практические задания для первого блока по базовому курсу по Python

## Задание 1: Анализ данных из текстового файла

In [3]:
import logging
import pandas as pd
from openpyxl import load_workbook

logging.basicConfig(
    filename='app_task1.log',
    filemode='w',
    level=logging.DEBUG,
    format='%(asctime)s - %(levelname)s - %(message)s',
    force=True,
    encoding='UTF-8',
)


def count_product(path_txt_file):
    """ Функция обработки данных из файла.

    Читает данные их файла data.txt, преобразует данные из списка в словарь, проводит расчеты дохода по товарам и общего дохода.
    Возвращает словарь и общий доход.
    """

    with open(path_txt_file, 'r', encoding='UTF-8') as file:
        content = file.readlines()
        logging.info('Чтение файла прошло успешно')
        products_data = {}

        for row in content:
            product, count, price = row.strip().split(',')
            products_data[product] ={
                'count': int(count),
                'price': float(price),
            }
        logging.info(f'Обработано товаров: {len(products_data)}')
        total_income = 0

        for data in products_data.values():
            total_revenue_product = data['count'] * data['price']
            data['total_revenue_product'] = round(total_revenue_product, 2)
            total_income += data['total_revenue_product']
            total_income = round(total_income, 2)

        logging.info(f'Расчет дохода завершен. Он составляет : {total_income}')

        return products_data, total_income


def create_excel_table(products_data, total_income):
    """ Функция создания excel таблицы с результатами расчета по данным файла data.txt .

    Принимает словарь с данными и общий доход, формирует DataFrame из списка, записывает DataFrame в excel файл.
    Возвращает excel таблицу с комментарием после последний сроки о результатах общих продаж.
    """

    logging.info('Создание Excel таблицы')

    # Создаем DataFrame
    df = pd.DataFrame.from_dict(
        products_data,
        orient='index'
    )

    df.index.name = 'Товар'

    df = df.rename(columns={
    'count': 'Количество',
    'price': 'Цена',
    'total_revenue_product': 'Выручка'
    }
    )

    #  Создаем и заполняем Excel файл
    with pd.ExcelWriter('table_output_task1.xlsx') as writer:
        df.to_excel(writer, index=True)

    wb = load_workbook('table_output_task1.xlsx')
    sheet = wb.active


    if total_income > 1000000:
        message = f'Общий доход равен : {total_income}, хорошо заработали))'
    else:
        message = f'Общий доход равен : {total_income}, в следующий раз надо постараться'

    message_row = sheet.max_row + 1

    sheet.cell(
        row=message_row,  # Строка для сообщения
        column=1,  # Колонка
        value=message,  # Сообщение
    )

    wb.save('table_output_task1.xlsx')
    logging.info('Excel файл сохранен успешно')

    # Читаем excel файл и возвращаем результат
    with pd.ExcelFile('table_output_task1.xlsx') as xls:
        df = pd.read_excel(xls, 'Sheet1')

    return (df)


path_txt_file = 'data_task1.txt'

try:
    products_data, total_income = count_product(path_txt_file)
    print(create_excel_table(products_data, total_income))

except FileNotFoundError as e:
    logging.critical(f'Ошибка, файл не найден: {e}')
except TypeError as e:
    logging.critical(f'Ошибка обработки данных: {e}')
except ValueError as e:
    logging.error(f'Ошибка формата данных: {e}')



                                               Товар  Количество      Цена  \
0                                          Телевизор         5.0  29999.99   
1                                        Холодильник         2.0  49999.50   
2                                           Смартфон        10.0  19999.00   
3                                            Ноутбук         3.0  69999.99   
4                                            Пылесос         7.0   8999.90   
5                                  Стиральная машина         4.0  35999.95   
6                                      Микроволновка         8.0   7999.99   
7                                          Кофеварка        15.0   4999.50   
8  Общий доход равен : 1005980.44, хорошо заработ...         NaN       NaN   

     Выручка  
0  149999.95  
1   99999.00  
2  199990.00  
3  209999.97  
4   62999.30  
5  143999.80  
6   63999.92  
7   74992.50  
8        NaN  


## Задание 2: Обработка данных из текстового файла

In [6]:
import logging


logging.basicConfig(
    filename='task2.log',
    filemode='w',
    level=logging.DEBUG,
    format='%(asctime)s - %(levelname)s - %(message)s',
    force=True,
    encoding='UTF-8',
)

def average_score(students) -> dict:
    """Функция принимает на вход файл со списком студентов и их оценками
    Обрабатывает данные из файла, расчитывает средний балл каждого студента
    Возовращает словарь с данными о студентах
    """
    
    with open(students, 'r', encoding='UTF-8') as file:
        content = file.readlines()
        logging.info('Чтение файла завершено успешно')
        students_data = {}

        logging.info('Обработка данных...')
        for row in content:
            student, *scores = row.strip().split(',')
            students_data[student] = {
                'scores': [int(grade) for grade in scores],
            }

            #  Расчет среднего балла каждого студента.
            students_data[student]['average score'] = sum(students_data[student]['scores']) / len(students_data[student]['scores'])
        
        logging.info(f'Данные в файле обоработаны успешно, посчитано {len(students_data.keys())} студентов')
        return students_data


def top_students(score_students: dict):
    """Функция принимает на вход словарь с данными о студентах
    Сортирует данные в файле по среднему баллу от большего к меньшему
    Записывает в новый файл студентов и их балл у которых средний балл выше threshold_top
    """

    threshold_top = 4.5
    
    logging.info('Сортировка студентов по среднему баллу...')
    score_students = sorted(score_students.items(),
                            key=lambda data: data[1]['average score'],  #  Передан ключ который сообщай о том , что нужно сортировать по среднему баллу.
                            reverse=True  #  Перевернули.
                           )
    logging.info('Сортировка прошла успешно')
    
    with open('top_students.txt', 'w', encoding='UTF-8') as top_file:
        logging.info('Начинает запись данных в файл top_students.txt')
        
        top_file.write(f'Топ студентов: \n')
        count_top_students = 0

        for student, data in score_students:
            if data['average score'] > threshold_top:
                count_top_students += 1
                top_file.write(f'{count_top_students}. {student} - средний балл: {data["average score"]}\n')
                

        logging.info(f'Запись лучших студентов с баллом выше {threshold_top} завершена. В топ лучших вошло {count_top_students} студента')


path_txt_file = 'students.txt'

try:
    top_students(average_score(path_txt_file))
    
    with open('top_students.txt', 'r', encoding='UTF-8') as top_students:
        top = top_students.read()
        print(top)

except FileNotFoundError as e:
    logging.critical(f'Ошибка, файл не найден: {e}')
except TypeError as e:
    logging.critical(f'Ошибка обработки данных: {e}')
except ValueError as e:
    logging.error(f'Ошибка формата данных: {e}')

Топ студентов: 
1. Анна Соколова - средний балл: 5.0
2. Елена Кузнецова - средний балл: 4.6
3. Мария Козлова - средний балл: 4.6
4. Сергей Новиков - средний балл: 4.6



## Задание 3: Создание простой базы данных в текстовом файле

In [ ]:
import logging

logging.basicConfig(
    filename='task2.log',
    filemode='w',
    level=logging.DEBUG,
    format='%(asctime)s - %(levelname)s - %(message)s',
    force=True,
    encoding='UTF-8',
)


class To_Do_list():
    def __init__(self):
        self.__to_do_list = []
        
        with open('tasks.txt', 'r', encoding='UTF-8') as tasks:
            file_tasks = tasks.readlines()
            for task in file_tasks:
                task = task.strip()
                if ': ' in task:
                    task = task.split(': ', 1)[1]
                    self.__to_do_list.append(task)
                elif task:
                    self.__to_do_list.append(task)
                    
    def save_tasks(self):
        with open('tasks.txt', 'w', encoding='UTF-8') as tasks:
            for number, task in enumerate(self.__to_do_list, start=1):
                tasks.write(f'{number}: {task}\n')                  

    @property
    def get_task(self):
        with open('tasks.txt', 'r', encoding='UTF-8') as tasks:
            file_tasks = tasks.read()
            return file_tasks

    
    def post_task(self, new_task: str):
        if isinstance(new_task, str):
            self.__to_do_list.append(new_task)
            with open('tasks.txt', 'a', encoding='UTF-8') as tasks:
                number_task = len(self.__to_do_list)
                tasks.write(f'{number_task}: {new_task}\n')
        else:
            raise TypeError('Отсутсвует либо некорректный тип данных задачи')

    
    def put_task(self, new_task: str):
        if isinstance(new_task, str):
            self.__to_do_list.clear()
            self.__to_do_list.append(new_task)
            self.save_tasks()
        else:
            raise TypeError('Отсутсвует либо некорректный тип данных задачи')

    
    def patch_task(self, id_task:int, new_task: str):
        if 1 <= id_task <= len(self.__to_do_list):
            self.__to_do_list[id_task - 1] = new_task
        else:
            raise ValueError('Задачи с таким номером нет')
        
        self.save_tasks()
    
    def dellete_task(self, id_task=None, task=None):
        if 1 <= id_task <= len(self.__to_do_list):
            del self.__to_do_list[id_task - 1]
            self.save_tasks()
                    
        else:
            raise TypeError('Задача для удаления отсутвует или неверного типа данных')


try:
    todo = To_Do_list()
    todo.save_tasks()

    while True:
        user_input = int(input('Введите цифру задачи операции которую хотите выполнить?\n 1) Добавить задачу\n 2) Удалить все задачи и добавить новую\n 3) Удалить задачу\n 4) Заменить задачу\n 5) Вывести весь список задач\n 6) Выйти\n : '))
        
        if user_input == 1:
            add_input =input('Введите текст задачи: ')
            todo.post_task(add_input)
            print(todo.get_task)

        elif user_input == 2:
            add_delall_input =input('Введите текст новой задачи (все остальные будут удалены): ')
            todo.put_task(add_delall_input)
            print(todo.get_task)

        elif user_input == 3:
            print(todo.get_task)
            del_input = int(input('Введите номер задачи которую хотите удалить: '))
            todo.dellete_task(del_input)
            print(todo.get_task)
            
        elif user_input == 4:
            patch_input = int(input('Введите номер задачи которую хотите заменить: '))
            new_task = input('Введите текст новой задачи: ')
            todo.patch_task(patch_input, new_task)
            print(todo.get_task)

        elif user_input == 5:
            print(todo.get_task)

        else:
            break

except FileNotFoundError as e:
    logging.critical(f'Ошибка, файл не найден: {e}')
except TypeError as e:
    logging.critical(f'Ошибка обработки данных: {e}')
except ValueError as e:
    logging.error(f'Ошибка формата данных: {e}')
        

Введите цифру задачи операции которую хотите выполнить?
 1) Добавить задачу
 2) Удалить все задачи и добавить новую
 3) Удалить задачу
 4) Заменить задачу
 5) Вывести весь список задач
 6) Выйти
 :  5


1: Купить продукты на ужин
2: Позвонить в больницу и записаться к врачу
3: Подготовить и отправить отчет директору
4: Сходить в спортзал – "качаем ноги"
5: Прочитать книгу "Понедельник начинается в субботу"
6: Очередной раз попытаться написать программу для решения "всех задач"
7: Убрать квартиру и приготовить ужин
8: Встретиться с друзьями и поиграть в настолки



Введите цифру задачи операции которую хотите выполнить?
 1) Добавить задачу
 2) Удалить все задачи и добавить новую
 3) Удалить задачу
 4) Заменить задачу
 5) Вывести весь список задач
 6) Выйти
 :  6
